In [ ]:
!pip install -U datasets

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from datasets import load_dataset

human_eval = load_dataset("openai/openai_humaneval")
# human_eval = load_dataset()
human_eval_renamed = load_dataset("csv", data_files="drive/MyDrive/lab/human_eval_renamed.csv")
human_eval["renamed"] = human_eval_renamed["train"]

print(human_eval)

In [ ]:
BAIDUBCE_API_KEY = "bce-v3/ALTAK-WLeFR782tNYNsNLMm0AJM/ca7b081a0d6cdabec651b076666e2461d762c27b"
SET_NAME = "renamed" # "test" | "renamed"
MODELS = [
    # "openai/gpt-4.1",
    # "openai/gpt-4.1-mini",
    # "openai/gpt-4.1-nano",
    # "openai/gpt-4o-mini",
    # "openai/gpt-4o-2024-11-20",
    # "deepseek/deepseek-chat-v3-0324",
    # "deepseek/deepseek-r1-0528",
    # "google/gemini-2.5-flash",
    # "google/gemini-2.5-pro",
    # "meta-llama/llama-3.3-70b-instruct",
    # "meta-llama/llama-3.1-8b-instruct",
    # "meta-llama/llama-3.2-3b-instruct",
    # "meta-llama/llama-3.2-1b-instruct",
    # "meta-llama/llama-4-maverick",
    # "meta-llama/llama-4-scout",
    "qwen/qwen3-30b-a3b",
    "qwen/qwen3-32b",
    "qwen/qwen3-8b",
    "qwen/qwen3-1.7b",
    "baidu/ernie-4.5-turbo-128k",
    "baidu/ernie-4.5-0.3b",
    "baidu/ernie-4.5-21b-a3b"
]

In [ ]:
from openai import OpenAI
import pickle
import os
from tqdm.notebook import tqdm

for model in MODELS:
    client = OpenAI(
        base_url="https://qianfan.baidubce.com/v2",
        api_key=BAIDUBCE_API_KEY
    )

    MODEL_PATH = model
    MODEL_NAME = model.split("/")[-1]

    # encapsulate openai API into a function that only requires messages as input
    def pipe(messages):
        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=messages
        )
        return response

    # messages = [
    #     {"role": "system", "content": "You are a helpful assistant."},
    #     {
    #         "role": "user",
    #         "content": "Hi Gemini"
    #     }
    # ]
    # print(pipe(messages))

    outputs = []
    PATH = f"drive/MyDrive/lab/{SET_NAME}/{MODEL_NAME.split('/')[-1]}-{SET_NAME}.pkl"
    if os.path.exists(PATH):
      with open(PATH, "rb") as f:
        outputs = pickle.load(f)

    for i in tqdm(range(len(outputs), len(human_eval[SET_NAME])), desc=f"{MODEL_PATH}-{SET_NAME}"):
        sample = human_eval[SET_NAME][i]

        prompt = sample["prompt"]
        messages = [
            {"role": "system", "content": "You are a code generation assistant. You are given the beginning part of the code and docstring. Complete the code without repeating given part, without any introductory or concluding remarks"},
            {"role": "user", "content": prompt},
        ]
        response = pipe(messages)
        outputs.append({
            "task_id": sample["task_id"],
            "prompt": prompt,
            'canonical_solution': sample["canonical_solution"],
            'test': sample["test"],
            'entry_point': sample["entry_point"],
            # "response": response[0]["generated_text"]
            "response": response.choices[0].message.content
        })
        with open(PATH, "wb") as f:
            pickle.dump(outputs, f)

        # rate limit: 180 requests per 10s interval
        # time.sleep(0.2)


In [ ]:
code = outputs[0]["response"]
# strip other text from code
code = code.split("```python")[1].split("```")[0]

test_code = outputs[0]["test"]

executes = code + "\n" + test_code + "\n" + "check({})".format(outputs[0]["entry_point"])
exec(executes)
print("PASS!")

In [ ]:
# run code as python script
code = """
from typing import List

def has_close_elements(numbers: List[float], threshold: float) -> bool:
    for idx, elem in enumerate(numbers):
        for idx2, elem2 in enumerate(numbers):
            if idx != idx2:
                distance = abs(elem - elem2)
                if distance < threshold:
                    return False    # False to trigger AssertionError

    return False
"""
test_code = outputs[0]["test"]
print(test_code)